# Boris pusher — C++ trajectories, Python tests (3D)

Split of responsibilities:
- **C++** (`src/`, `tools/boris_traj.cpp`): the non-relativistic pusher + a trajectory-only driver.
- **This notebook**: analytic reference solutions, 3D plots, and all PASS/FAIL tests.

Run top to bottom.

In [ ]:
# Build the C++ trajectory driver
!cmake -S . -B build > /dev/null && cmake --build build > /dev/null && mkdir -p data && ls build/boris_traj

In [ ]:
%matplotlib inline
import subprocess
import numpy as np
import matplotlib.pyplot as plt

BIN = './build/boris_traj'

# --- Analytic reference solutions (non-relativistic, uniform static fields) ---
def pure_B(x0, v0, q, m, B0, t):
    t = np.asarray(t, dtype=float)
    W = q * B0 / m  # gyro-frequency
    c, s = np.cos(W * t), np.sin(W * t)
    vx = v0[0] * c + v0[1] * s
    vy = v0[1] * c - v0[0] * s
    vz = np.full_like(t, v0[2])
    x = x0[0] + (v0[0] * s + v0[1] * (1 - c)) / W
    y = x0[1] + (v0[1] * s - v0[0] * (1 - c)) / W
    z = x0[2] + v0[2] * t
    return np.stack([x, y, z], -1), np.stack([vx, vy, vz], -1)

def e_cross_b(x0, v0, q, m, E0, B0, t):
    vd = E0 / B0  # E x B / B^2 = (vd, 0, 0)
    w0 = (v0[0] - vd, v0[1], v0[2])
    xw, vw = pure_B((0, 0, 0), w0, q, m, B0, t)
    t = np.asarray(t, dtype=float)
    xa = np.stack([x0[0] + vd * t + xw[:, 0], x0[1] + xw[:, 1], x0[2] + xw[:, 2]], -1)
    va = np.stack([vd + vw[:, 0], vw[:, 1], vw[:, 2]], -1)
    return xa, va

def e_parallel_b(x0, v0, q, m, E0, B0, t):
    xp, vp = pure_B((x0[0], x0[1], 0), (v0[0], v0[1], 0), q, m, B0, t)
    t = np.asarray(t, dtype=float)
    az = q * E0 / m
    xa = np.stack([xp[:, 0], xp[:, 1], x0[2] + v0[2] * t + 0.5 * az * t**2], -1)
    va = np.stack([vp[:, 0], vp[:, 1], v0[2] + az * t], -1)
    return xa, va

# --- Helpers: run C++ driver, compare against analytics ---
def run_case(name, x0, v0, E, B, q=1.0, m=1.0, dt=0.01, n=100):
    """Run the C++ pusher, return trajectory array (boris only)."""
    out = f'data/{name}.csv'
    cmd = [BIN, out, *map(str, [*x0, *v0, *E, *B, q, m, dt, n])]
    subprocess.run(cmd, check=True)
    return np.genfromtxt(out, delimiter=',', names=True)

results = []  # (name, ok)

def check(name, err, tol):
    ok = err < tol
    results.append((name, ok))
    print(f'  {name:22s} err={err:.3e} tol={tol:.1e}  {"PASS" if ok else "FAIL"}')
    return ok

In [ ]:
# 1) Pure B: gyro-orbit (circle in xy-plane)
x0, v0, q, m, B0 = (0, 0, 0), (1, 0, 0), 1.0, 1.0, 1.0
T = 2 * np.pi  # Omega = qB/m = 1
pure = run_case('pure_b', x0, v0, (0, 0, 0), (0, 0, B0), q, m, T / 200, 200)
xa, va = pure_B(x0, v0, q, m, B0, pure['t'])

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot(xa[:, 0], xa[:, 1], xa[:, 2], label='analytic')
ax.plot(pure['x'], pure['y'], pure['z'], '--', label='boris (C++)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Pure B (B=1z): gyro-orbit')
ax.legend()
plt.show()

In [ ]:
# 2) E perp B: cycloid drifting along +x (vd = E x B / B^2)
x0, v0, q, m, E0, B0 = (0, 0, 0), (0, 0, 0), 1.0, 1.0, 1.0, 1.0
exb = run_case('exb', x0, v0, (0, E0, 0), (0, 0, B0), q, m, T / 200, 200)
xa, va = e_cross_b(x0, v0, q, m, E0, B0, exb['t'])

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot(xa[:, 0], xa[:, 1], xa[:, 2], label='analytic')
ax.plot(exb['x'], exb['y'], exb['z'], '--', label='boris (C++)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('E perp B (E=1y, B=1z): cycloid + drift')
ax.legend()
plt.show()

In [ ]:
# 3) E parallel B: helix with uniform acceleration along z
x0, v0, q, m, E0, B0 = (0, 0, 0), (1, 0, 0.5), 1.0, 1.0, 1.0, 1.0
epar = run_case('epar_b', x0, v0, (0, 0, E0), (0, 0, B0), q, m, T / 200, 200)
xa, va = e_parallel_b(x0, v0, q, m, E0, B0, epar['t'])

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot(xa[:, 0], xa[:, 1], xa[:, 2], label='analytic')
ax.plot(epar['x'], epar['y'], epar['z'], '--', label='boris (C++)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('E parallel B (E=B=1z): accelerated helix')
ax.legend()
plt.show()

In [ ]:
# TESTS: trajectory error, energy conservation, drift velocity
def errs(d, xa, va):
    b = np.stack([d['x'], d['y'], d['z']], -1)
    bv = np.stack([d['vx'], d['vy'], d['vz']], -1)
    return np.linalg.norm(b - xa, axis=1), np.linalg.norm(bv - va, axis=1)

print('[1] Pure B')
xa, va = pure_B((0, 0, 0), (1, 0, 0), 1, 1, 1.0, pure['t'])
xe, ve = errs(pure, xa, va)
check('max |x-xa|', xe.max(), 2e-3)
check('max |v-va|', ve.max(), 2e-3)
vn = np.sqrt(pure['vx']**2 + pure['vy']**2 + pure['vz']**2)
check('|v| conservation', np.abs(vn - vn[0]).max(), 1e-12)

print('[2] E perp B')
xa, va = e_cross_b((0, 0, 0), (0, 0, 0), 1, 1, 1.0, 1.0, exb['t'])
xe, ve = errs(exb, xa, va)
check('max |x-xa|', xe.max(), 2e-3)
check('max |v-va|', ve.max(), 2e-3)
check('<vx>-vd drift', abs(exb['vx'][1:].mean() - 1.0), 2e-3)

print('[3] E parallel B')
xa, va = e_parallel_b((0, 0, 0), (1, 0, 0.5), 1, 1, 1.0, 1.0, epar['t'])
xe, ve = errs(epar, xa, va)
check('max |x-xa|', xe.max(), 2e-3)
check('max |v-va|', ve.max(), 2e-3)

# Error growth over one gyroperiod
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for d, fn, args, name in [
        (pure, pure_B, ((0, 0, 0), (1, 0, 0), 1, 1, 1.0), 'pure B'),
        (exb, e_cross_b, ((0, 0, 0), (0, 0, 0), 1, 1, 1.0, 1.0), 'E perp B'),
        (epar, e_parallel_b, ((0, 0, 0), (1, 0, 0.5), 1, 1, 1.0, 1.0), 'E par B')]:
    xa, va = fn(*args, d['t'])
    xe, ve = errs(d, xa, va)
    axs[0].plot(d['t'], xe, label=name)
    axs[1].plot(d['t'], ve, label=name)
axs[0].set(xlabel='t', ylabel='|x - xa|', title='Position error'); axs[0].legend()
axs[1].set(xlabel='t', ylabel='|v - va|', title='Velocity error'); axs[1].legend()
plt.show()

In [ ]:
# TEST 4: convergence vs dt (pure B, error at t=T; expect slope ~2)
print('[4] Convergence vs dt')
divs = [25, 50, 100, 200, 400]
dts, errs = [], []
for k in divs:
    d = run_case(f'conv_{k}', (0, 0, 0), (1, 0, 0), (0, 0, 0), (0, 0, 1.0),
                 1.0, 1.0, T / k, k)
    xa, va = pure_B((0, 0, 0), (1, 0, 0), 1, 1, 1.0, T)
    e = np.linalg.norm([d['x'][-1] - xa[0], d['y'][-1] - xa[1], d['z'][-1] - xa[2]])
    dts.append(T / k); errs.append(e)
    print(f'  div={k:4d} dt={T / k:.3e}  |x-xa|={e:.3e}')
dts, errs = np.array(dts), np.array(errs)
slopes = np.log(errs[:-1] / errs[1:]) / np.log(2)
print('  slopes:', ' '.join(f'{s:.3f}' for s in slopes))
check('2nd-order slopes', max(abs(slopes - 2.0).max(), 0.0), 0.3)

slope = np.polyfit(np.log(dts), np.log(errs), 1)[0]
plt.loglog(dts, errs, 'o-', label=f'measured (slope={slope:.2f})')
plt.loglog(dts, errs[0] * (dts / dts[0])**2, '--', label='O(dt^2) reference')
plt.xlabel('dt'); plt.ylabel('|x - xa| at t=T')
plt.title('Convergence (pure B)')
plt.legend(); plt.show()

# Final verdict
n_fail = sum(not ok for _, ok in results)
print(f'\n{len(results) - n_fail}/{len(results)} checks passed')
assert n_fail == 0, f'{n_fail} check(s) FAILED'

## N particles: ensemble in a shared uniform field

C++ side (`src/ensemble.h`): the ensemble is a **struct-of-arrays** — one array per quantity
(`x[i]`, `v[i]`, `q[i]`, `m[i]` form particle `i`). `push_ensemble` loops over the trusted
single-particle `boris_push`, so the physics still lives in exactly one place.
`tools/boris_traj_n` advances the whole ensemble (CSV in → CSV out).

This section writes `data/ensemble_init.csv` (varied `v0` and `q/m`, incl. a negative charge),
runs **one** multi-particle binary call in pure B, then tests every particle against its own analytic solution.

In [ ]:
# N-particle run: 4 particles, shared uniform B=1z, one total time T = 2*pi
import csv
BIN_N = './build/boris_traj_n'
parts = [
    dict(x0=(0,0,0), v0=(1,0,0),     q=1.0,  m=1.0),  # reference: one full orbit
    dict(x0=(0,0,0), v0=(0,1,0),     q=1.0,  m=2.0),  # heavy (Om=0.5): half orbit
    dict(x0=(0,0,0), v0=(1,0,0.5),   q=-1.0, m=1.0),  # negative q: reversed gyration
    dict(x0=(0,0,0), v0=(0.5,0.5,0), q=2.0,  m=1.0),  # fast (Om=2): two orbits
]
Tn, nn = 2*np.pi, 200
with open('data/ensemble_init.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['x','y','z','vx','vy','vz','q','m'])
    for p in parts:
        w.writerow([*p['x0'], *p['v0'], p['q'], p['m']])
subprocess.run([BIN_N, 'data/ensemble.csv', 'data/ensemble_init.csv',
                '0','0','0', '0','0','1', str(Tn/nn), str(nn)], check=True)
ens = np.genfromtxt('data/ensemble.csv', delimiter=',', names=True)
print('particles:', sorted(set(ens['pid'])), 'rows:', len(ens))


In [ ]:
# 3D: all N trajectories (solid, C++) vs analytics (dashed)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
for i, p in enumerate(parts):
    d = ens[ens['pid'] == i]
    xa, va = pure_B(p['x0'], p['v0'], p['q'], p['m'], 1.0, d['t'])
    ax.plot(xa[:,0], xa[:,1], xa[:,2], '--')
    ax.plot(d['x'], d['y'], d['z'], label=f"p{i} q/m={p['q']/p['m']:+.1f}")
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Ensemble in pure B (shared field, per-particle q/m)')
ax.legend()
plt.show()


In [ ]:
# TEST 5: per-particle trajectory error + energy conservation
print('[5] Ensemble (pure B, shared field)')
results_n = []
def check_n(name, err, tol):
    ok = err < tol
    results_n.append((name, ok))
    print(f'  {name:22s} err={err:.3e} tol={tol:.1e}  {"PASS" if ok else "FAIL"}')
# Note: tol 5e-3 covers the fast particle (Om=2), whose gyroperiod is
# resolved 2x more coarsely at the shared dt — error scales ~(Om*dt)^2.
for i, p in enumerate(parts):
    d = ens[ens['pid'] == i]
    b = np.stack([d['x'], d['y'], d['z']], -1)
    bv = np.stack([d['vx'], d['vy'], d['vz']], -1)
    xa, va = pure_B(p['x0'], p['v0'], p['q'], p['m'], 1.0, d['t'])
    xe = np.linalg.norm(b - xa, axis=1).max()
    ve = np.linalg.norm(bv - va, axis=1).max()
    vn = np.linalg.norm(bv, axis=1)
    print(f'  -- p{i} (q/m={p["q"]/p["m"]:+.1f})')
    check_n(f'p{i} max |x-xa|', xe, 5e-3)
    check_n(f'p{i} max |v-va|', ve, 5e-3)
    check_n(f'p{i} |v| conserv.', np.abs(vn - vn[0]).max(), 1e-12)
n_fail_n = sum(not ok for _, ok in results_n)
print(f'\n{len(results_n) - n_fail_n}/{len(results_n)} ensemble checks passed')
assert n_fail_n == 0, f'{n_fail_n} ensemble check(s) FAILED'


## Non-uniform analytic field: linear B gradient

`B(x) = B0 (1 + αx) ẑ`, `E = 0` (`src/fields_model.h`, driver `boris_traj_field`).
Guiding-center theory predicts a grad-B drift in `+y`: `v_dy = m v⊥² α / (2 q B0)`.
Two subtleties this section handles explicitly:
- Particles are initialized so their **guiding centers sit at `x=0`** (where `B=B0`); otherwise the local
  gyroperiod differs from `2π` and both the theory comparison and the drift fit go wrong (~14% seen in testing).
- Drift is measured **stroboscopically** (`y` sampled at integer gyroperiods): a plain linear fit leaks
  gyro-oscillation (`r_L / T_window`) into the slope, which swamps slow drifts.

In [ ]:
# Non-uniform run: 2 particles, 10 gyro-orbits, shared grad-B field
BIN_F = './build/boris_traj_field'
B0g, ag = 1.0, 0.1
gparts = [
    dict(x0=(0, 0, 0),    v0=(0.5, 0, 0), q=1.0, m=1.0, vd=0.5**2*ag/2),  # vd=0.0125
    dict(x0=(-0.3, 0, 0), v0=(0, 0.3, 0), q=1.0, m=1.0, vd=0.3**2*ag/2),  # vd=0.0045
]
Tg, dtg, ng = 2*np.pi, 0.01, 6280
with open('data/grad_init.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['x','y','z','vx','vy','vz','q','m'])
    for p in gparts:
        w.writerow([*p['x0'], *p['v0'], p['q'], p['m']])
subprocess.run([BIN_F, 'data/grad.csv', 'data/grad_init.csv', 'lingrad',
                str(B0g), str(ag), str(dtg), str(ng)], check=True)
grad = np.genfromtxt('data/grad.csv', delimiter=',', names=True)
print('particles:', sorted(set(grad['pid'])), 'rows:', len(grad))


In [ ]:
# 3D: gyro-orbits visibly drifting in +y
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
for i, p in enumerate(gparts):
    d = grad[grad['pid'] == i]
    ax.plot(d['x'], d['y'], d['z'], label=f"p{i} v⊥={np.linalg.norm(p['v0'][:2]):.1f}")
    ax.scatter([d['x'][0]], [d['y'][0]], [d['z'][0]])  # start marker
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Linear grad-B: gyro-orbits drifting in +y')
ax.legend()
plt.show()


In [ ]:
# TEST 6a (exact): |v| conservation — B does no work, even non-uniformly
# TEST 6b (theory): stroboscopic drift vs v_dy = m v⊥² α/(2qB0)
print('[6] grad-B drift + energy')
results_g = []
def check_g(name, err, tol):
    ok = err < tol
    results_g.append((name, ok))
    print(f'  {name:22s} err={err:.3e} tol={tol:.1e}  {"PASS" if ok else "FAIL"}')
for i, p in enumerate(gparts):
    d = grad[grad['pid'] == i]
    vn = np.sqrt(d['vx']**2 + d['vy']**2 + d['vz']**2)
    print(f'  -- p{i} (v⊥={np.linalg.norm(p["v0"][:2]):.1f})')
    check_g(f'p{i} |v| conserv.', np.abs(vn - vn[0]).max(), 1e-12)
    ks = np.arange(1, int(d['t'][-1] / Tg) + 1)  # strobe at integer gyroperiods
    slope = np.polyfit(ks * Tg, np.interp(ks * Tg, d['t'], d['y']), 1)[0]
    print(f'    measured v_dy={slope:.6f} theory={p["vd"]:.6f}')
    check_g(f'p{i} drift vs theory', abs(slope - p['vd']) / p['vd'], 0.10)


In [ ]:
# TEST 6c (self-convergence): same trajectory at several dts over 2 orbits.
# Runs end at slightly different times (n*dt), so all are INTERPOLATED to
# a common time before comparing; errs use an extra-fine reference run.
print('[7] Self-convergence (grad-B, 2 orbits)')
dts = [0.04, 0.02, 0.01, 0.005, 0.0025]
trajs = []
for dt in dts:
    n = int(2 * Tg / dt)
    subprocess.run([BIN_F, 'data/grad_c.csv', 'data/grad_init.csv', 'lingrad',
                    str(B0g), str(ag), str(dt), str(n)],
                   check=True, capture_output=True)
    d = np.genfromtxt('data/grad_c.csv', delimiter=',', names=True)
    m = d[d['pid'] == 0]
    trajs.append((m['t'], np.stack([m['x'], m['y'], m['z']], -1)))
tc = min(t[-1] for t, _ in trajs)
ends = np.array([[[np.interp(tc, t, x[:, k]) for k in range(3)]]
                 for t, x in trajs])[:, 0, :]
errs = np.array([np.linalg.norm(e - ends[-1]) for e in ends[:-1]])
for dt, e in zip(dts[:-1], errs):
    print(f'  dt={dt:.3e}  |x(dt)-x(ref)|={e:.3e}')
pair = np.log(errs[:-1] / errs[1:]) / np.log(2)
fit = np.polyfit(np.log(np.array(dts[:-1])), np.log(errs), 1)[0]
print('  pairwise slopes:', ' '.join(f'{s:.3f}' for s in pair))
print(f'  fitted slope: {fit:.3f}')
ok = 1.7 < fit < 2.3
results_g.append(('self-conv 2nd order', ok))
print(f'  self-conv 2nd order  {"PASS" if ok else "FAIL"}')
plt.loglog(dts[:-1], errs, 'o-', label=f'measured (slope={fit:.2f})')
plt.loglog(dts[:-1], errs[0] * (np.array(dts[:-1]) / dts[0])**2, '--', label='O(dt^2)')
plt.xlabel('dt'); plt.ylabel('|x(dt) - x(ref)| at t=tc')
plt.title('Self-convergence (grad-B)'); plt.legend(); plt.show()
n_fail_g = sum(not ok for _, ok in results_g)
print(f'\n{len(results_g) - n_fail_g}/{len(results_g)} grad-B checks passed')
assert n_fail_g == 0, f'{n_fail_g} grad-B check(s) FAILED'


## Field grid + trilinear gather (final step)

C++ side (`src/field_grid.h`): `FieldGrid` stores `E`/`B` on a uniform 3D node mesh over a
periodic domain; `sample(x)` gathers via **trilinear interpolation in the `c0..c7` coefficient
form of our Python prototype** (same corner order, same coefficients — see `tri8`).
`push_ensemble_grid` evaluates at the step midpoint and wraps positions back into the domain.

Key facts this section establishes:
- Trilinear interpolation is **exact for (multi)linear fields** — our `lingrad` field qualifies,
  so grid-vs-analytic agreement to machine precision is expected (and proves indexing/gather correct).
- A `uniform`-filled grid has a periodic extension equal to the analytic field *everywhere*,
  isolating the **wrap logic**: a boundary-crossing orbit must match the folded analytic one.
- Drift theory must use the **local** `B` at the guiding center: `v_dy = m v⊥² B0 α / (2qB²)`.

In [ ]:
# Grid run: lingrad field sampled onto a 32^3 grid over [0,4)^3 (10 orbits)
BIN_G = './build/boris_traj_grid'
Lq, nq = 4.0, 32
qparts = [
    dict(x0=(2.0, 2, 0), v0=(0.5, 0, 0), q=1.0, m=1.0, Xgc=2.0, vperp=0.5),
    dict(x0=(1.7, 2, 0), v0=(0, 0.3, 0), q=1.0, m=1.0, Xgc=2.0, vperp=0.3),
]
B0q, aq = 1.0, 0.1
Tq, dtq, nq_steps = 2*np.pi, 0.01, 6280
with open('data/grid_init.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['x','y','z','vx','vy','vz','q','m'])
    for p in qparts:
        w.writerow([*p['x0'], *p['v0'], p['q'], p['m']])
subprocess.run([BIN_G, 'data/grid.csv', 'data/grid_init.csv', 'lingrad',
                str(B0q), str(aq), str(nq), '0', '0', '0',
                str(Lq), str(dtq), str(nq_steps)], check=True)
gr = np.genfromtxt('data/grid.csv', delimiter=',', names=True)
# Analytic-driver reference on the SAME initial conditions
subprocess.run(['./build/boris_traj_field', 'data/grid_ref.csv', 'data/grid_init.csv',
                'lingrad', str(B0q), str(aq), str(dtq), str(nq_steps)],
               check=True, capture_output=True)
gr_ref = np.genfromtxt('data/grid_ref.csv', delimiter=',', names=True)
print('particles:', sorted(set(gr['pid'])), 'rows:', len(gr))


In [ ]:
# 3D: grid-gathered (solid) vs analytic-field (dashed) trajectories
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
for i, p in enumerate(qparts):
    d = gr[gr['pid'] == i]; r = gr_ref[gr_ref['pid'] == i]
    ax.plot(r['x'], r['y'], r['z'], '--')
    ax.plot(d['x'], d['y'], d['z'], label=f"p{i} v⊥={p['vperp']}")
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Grid gather (32^3) vs analytic field')
ax.legend()
plt.show()


In [ ]:
# TEST 8: grid-vs-analytic exactness, energy, drift with LOCAL B, dx-flatness
print('[8] Grid gather (lingrad, 32^3)')
results_q = []
def check_q(name, err, tol):
    ok = err < tol
    results_q.append((name, ok))
    print(f'  {name:22s} err={err:.3e} tol={tol:.1e}  {"PASS" if ok else "FAIL"}')
for i, p in enumerate(qparts):
    d = gr[gr['pid'] == i]; r = gr_ref[gr_ref['pid'] == i]
    xe = np.linalg.norm(np.stack([d['x'], d['y'], d['z']], -1)
                        - np.stack([r['x'], r['y'], r['z']], -1), axis=1).max()
    vn = np.sqrt(d['vx']**2 + d['vy']**2 + d['vz']**2)
    print(f'  -- p{i} (v⊥={p["vperp"]})')
    check_q(f'p{i} grid-vs-analytic', xe, 1e-10)
    check_q(f'p{i} |v| conserv.', np.abs(vn - vn[0]).max(), 1e-12)
    Bgc = B0q * (1 + aq * p['Xgc'])  # LOCAL field at the guiding center
    vd = p['m'] * p['vperp']**2 * B0q * aq / (2 * p['q'] * Bgc**2)
    Om = p['q'] * Bgc / p['m']
    A = np.stack([d['t'], np.ones(len(d)), np.sin(Om*d['t']), np.cos(Om*d['t'])], -1)
    slope = np.linalg.lstsq(A, d['y'], rcond=None)[0][0]  # drift + gyration joint fit
    print(f'    measured v_dy={slope:.6f} theory={vd:.6f}')
    check_q(f'p{i} drift vs theory', abs(slope - vd) / vd, 0.10)
print('dx-flatness (linear B interpolates exactly at any dx):')
for nx in [8, 16, 32, 64]:
    subprocess.run([BIN_G, 'data/grid_dx.csv', 'data/grid_init.csv', 'lingrad',
                    str(B0q), str(aq), str(nx), '0', '0', '0',
                    str(Lq), str(dtq), '628'], check=True, capture_output=True)
    subprocess.run(['./build/boris_traj_field', 'data/grid_dx_ref.csv', 'data/grid_init.csv',
                    'lingrad', str(B0q), str(aq), str(dtq), '628'],
                   check=True, capture_output=True)
    d = np.genfromtxt('data/grid_dx.csv', delimiter=',', names=True)
    r = np.genfromtxt('data/grid_dx_ref.csv', delimiter=',', names=True)
    m0 = d[d['pid'] == 0]; r0 = r[r['pid'] == 0]
    e = np.linalg.norm(np.stack([m0['x'], m0['y'], m0['z']], -1)
                       - np.stack([r0['x'], r0['y'], r0['z']], -1), axis=1).max()
    print(f'  nx={nx:3d} (dx={Lq/nx:.4f})  grid-vs-analytic={e:.3e}')
    check_q(f'nx={nx} exactness', e, 1e-10)


In [ ]:
# TEST 9: periodic wrap — uniform grid crosser vs FOLDED analytic orbit.
# A uniform grid's periodic extension equals the analytic field everywhere,
# so this isolates gather-across-boundary + particle wrap (each must be exact).
print('[9] Periodic wrap (uniform grid, boundary crosser)')
with open('data/wrap_init.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['x','y','z','vx','vy','vz','q','m'])
    w.writerow([3.9, 0, 0, 0.5, 0, 0, 1.0, 1.0])  # orbit spans x in [3.4, 4.4]
subprocess.run([BIN_G, 'data/wrap.csv', 'data/wrap_init.csv', 'uniform',
                '1.0', '0', '16', '0', '0', '0', str(Lq), '0.01', '1256'], check=True)
wp = np.genfromtxt('data/wrap.csv', delimiter=',', names=True)
t = wp['t']
fold = lambda v: np.mod(v, Lq)
xa, ya, za = fold(3.9 + 0.5*np.sin(t)), fold(-0.5*(1-np.cos(t))), fold(np.zeros_like(t))
dd = np.abs(np.stack([wp['x'], wp['y'], wp['z']], -1) - np.stack([xa, ya, za], -1))
xe = np.linalg.norm(np.minimum(dd, Lq - dd), axis=1).max()  # minimal image
print(f'  boundary crossings: {(np.abs(np.diff(wp["x"])) > Lq/2).sum()}')
print(f'  stays in domain: {wp["x"].min() >= 0 and wp["x"].max() < Lq}')
check_q('wrap folded err', xe, 2e-3)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot(xa, ya, za, '--', label='analytic (folded)')
ax.plot(wp['x'], wp['y'], wp['z'], label='grid (wrapped)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Boundary crosser: wrap vs folded analytic')
ax.legend(); plt.show()
n_fail_q = sum(not ok for _, ok in results_q)
print(f'\n{len(results_q) - n_fail_q}/{len(results_q)} grid checks passed')
assert n_fail_q == 0, f'{n_fail_q} grid check(s) FAILED'


### Optimization notes (gather cost)

Measured in this step: the grid runs reproduce analytic trajectories to machine precision at any
`dx`, so all remaining error is pusher error — the gather itself adds no physics error for this field class.

Cost breakdown per particle-step (serial, as implemented):
- `sample()`: 8 nodes × 6 components = **48 doubles loaded**, ~100 flops of `tri8` math → **memory-bound**.
- Midpoint scheme: 1 gather + 1 push per step (the push needs no second gather).
- SoA layout (`x`/`v`/`q`/`m` arrays) streams linearly through cache — the right layout if this ever scales.
- Cheap wins if N grows: hoist `1/dx` reciprocals (already divisions per sample), store E/B interleaved
  per node for one cache line per corner, then `#pragma omp parallel for` over the ensemble loop
  (iterations are independent — shared read-only grid). None of that is needed at notebook scale.